In [ ]:
from scipy import stats
import numpy as np
import json

# Obtain test results

In [ ]:
def get_results(filepath):
    """ Load results from JSON file. """
    with open(filepath, 'r') as file:
        data = json.load(file)
    return data

def compute_ci(values, ci=0.95):
    n = len(values)
    mean = np.mean(values)
    std = np.std(values, ddof=1)
    t_val = stats.t.ppf((1 + ci) / 2, df=n-1)
    margin = t_val * std / np.sqrt(n)
    return mean - margin, mean + margin


In [ ]:
def obtain_surv_test_results(type_of_survival, exp_name, type=None):
    """ Print test results for a specific type of survival and experiment name. """
    # Skip these results
    list_keys = ["c_index", "c_index_ipcw"]
    fold_res_file = f"results/{type_of_survival}/{exp_name}/Results.json"
    result_data = get_results(fold_res_file)

    # Store results over all folds in one list
    all_results = []
    for key, value in result_data.items():
        all_results.append(value)
        
    # Combine results for each key over all folds
    keys = value.keys()
    combined_results_data = {key: np.array([d[key] for d in all_results]) for key in keys}
    means = {key: np.mean(values) for key, values in combined_results_data.items()}
    stds = {key: np.std(values) for key, values in combined_results_data.items()} 

    print(f"Results of {type_of_survival} for experiment {exp_name}:")     
    for key in combined_results_data:
        if key in list_keys:
            mean = means[key]
            # std = stds[key]
            lower, upper = compute_ci(combined_results_data[key])
            print(f"{key}: {mean:.3f} [{lower:.3f}, {upper:.3f}]")

def obtain_test_results(type_of_survival, exp_name, type=None):
    """ Print test results for a specific type of survival and experiment name. """
    # Skip these results
    list_non_keys = ["loss", "survival_loss", "disentanglement_loss", "disentanglement_multi_loss", "disentanglement_single_loss", "disentanglement_D1_loss", "disentanglement_D2_loss"]
    fold_res_file = f"results/{type_of_survival}/{exp_name}/Results.json"
    result_data = get_results(fold_res_file)

    # Store results over all folds in one list
    all_results = []
    for key, value in result_data.items():
        all_results.append(value)
        
    # Combine results for each key over all folds
    keys = value.keys()
    combined_results_data = {key: np.array([d[key] for d in all_results]) for key in keys}
    means = {key: np.mean(values) for key, values in combined_results_data.items()}
    stds = {key: np.std(values) for key, values in combined_results_data.items()} 

    

    if type:
        mean = means[type]
        lower, upper = compute_ci(combined_results_data[type])
        print(f"{type}: {mean:.3f}$_" + "{\\text{["+f"{lower:.3f}, {upper:.3f}"+"]}}$")
        return 
    
    for key in combined_results_data:
        if not key in list_non_keys:
            mean = means[key]
            lower, upper = compute_ci(combined_results_data[key])
            print(f"{key}: {mean:.3f}$_" + "{\\text{["+f"{lower:.3f}, {upper:.3f}"+"]}}$")
            # std = stds[key]
            # print(f"{key}: {mean:.3f}±{std:.3f}")

In [ ]:
# Print results
obtain_test_results(type_of_survival='dss_survival_brca', exp_name='DIMAFx')